# 04 - Multi-Source Knowledge Base 구성

이 노트북에서는 **여러 데이터 소스**를 통합한 Knowledge Base를 구성합니다.

## 📋 학습 내용
1. **다양한 Knowledge Source 유형** 이해
   - `searchIndex` - Azure AI Search 인덱스
   - `azureBlob` - Azure Blob Storage
   - `web` - 웹 URL 크롤링
   - `remoteSharePoint` - SharePoint Online
   - `indexedOneLake` - Microsoft Fabric OneLake
2. 여러 소스를 하나의 Knowledge Base로 통합
3. 소스별 우선순위 및 설정 조정

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료
- Azure Blob Storage 컨테이너 (선택사항)

## 1. 환경 설정

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# Azure OpenAI 설정
aoai_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
chat_deployment = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

# 클라이언트 생성
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)

print(f"✅ Search Endpoint: {search_endpoint}")
print(f"✅ OpenAI Endpoint: {aoai_endpoint}")

## 2. Knowledge Source 유형 이해

FoundryIQ는 다양한 데이터 소스를 Knowledge Source로 연결할 수 있습니다:

| 소스 유형 | 설명 | 사용 사례 |
|----------|------|----------|
| `searchIndex` | 기존 Azure AI Search 인덱스 | 이미 인덱싱된 데이터 활용 |
| `azureBlob` | Azure Blob Storage | PDF, 문서 파일 |
| `web` | 웹 URL | 온라인 문서, 기술 문서 |
| `remoteSharePoint` | SharePoint Online | 기업 문서 |
| `indexedSharePoint` | SharePoint (인덱싱됨) | 대량 SharePoint 문서 |
| `indexedOneLake` | Microsoft Fabric OneLake | 데이터 레이크 |

## 3. 기존 Knowledge Source 확인

In [ ]:
# 기존 Knowledge Sources 목록 조회
print("📋 기존 Knowledge Sources:")
print("=" * 50)

try:
    knowledge_sources = list(index_client.list_knowledge_sources())
    
    if knowledge_sources:
        for ks in knowledge_sources:
            print(f"\n🔗 이름: {ks.name}")
            print(f"   설명: {ks.description or 'N/A'}")
            print(f"   유형: {type(ks.data).__name__}")
    else:
        print("등록된 Knowledge Source가 없습니다.")
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 4. 추가 Knowledge Source 생성

### 4.1 Web URL Knowledge Source

웹 문서를 Knowledge Source로 연결합니다. 예: 기술 문서, API 레퍼런스

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeSource,
    WebKnowledgeSourceData,
)

# Web Knowledge Source 정의
# 참고: 실제 운영 시에는 접근 가능한 URL을 사용해야 합니다
WEB_SOURCE_NAME = "docs-web-source"

web_knowledge_source = KnowledgeSource(
    name=WEB_SOURCE_NAME,
    description="Azure AI Search 공식 문서 (웹 크롤링)",
    data=WebKnowledgeSourceData(
        urls=[
            "https://learn.microsoft.com/azure/search/search-what-is-azure-search",
            "https://learn.microsoft.com/azure/search/search-get-started-portal"
        ]
    )
)

print(f"📌 Web Knowledge Source 정의:")
print(f"   이름: {WEB_SOURCE_NAME}")
print(f"   URL 수: {len(web_knowledge_source.data.urls)}")
print(f"\n⚠️ 참고: Web 소스는 Answer Synthesis 모드에서만 사용 가능합니다.")

In [ ]:
# Web Knowledge Source 생성 (선택적 - URL 접근 가능 시에만)
# 주석을 해제하여 실행하세요

# try:
#     result = index_client.create_or_update_knowledge_source(web_knowledge_source)
#     print(f"✅ Web Knowledge Source '{result.name}' 생성 완료")
# except Exception as e:
#     print(f"❌ 생성 실패: {e}")

print("ℹ️ Web Knowledge Source 생성은 선택사항입니다.")
print("   실제 환경에서 접근 가능한 URL로 설정 후 주석을 해제하세요.")

### 4.2 Azure Blob Storage Knowledge Source

Azure Blob Storage의 문서를 Knowledge Source로 연결합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSourceData,
)

# Azure Blob Storage 설정 (환경 변수에서 로드, 없으면 예시값)
STORAGE_ACCOUNT = os.environ.get("STORAGE_ACCOUNT_NAME", "your-storage-account")
STORAGE_CONTAINER = os.environ.get("STORAGE_CONTAINER_NAME", "documents")
BLOB_SOURCE_NAME = "blob-documents-source"

# Blob Knowledge Source 정의
blob_knowledge_source = KnowledgeSource(
    name=BLOB_SOURCE_NAME,
    description="Azure Blob Storage의 PDF 및 문서 파일",
    data=AzureBlobKnowledgeSourceData(
        storage_account_resource_id=f"/subscriptions/{{subscription_id}}/resourceGroups/{{rg_name}}/providers/Microsoft.Storage/storageAccounts/{STORAGE_ACCOUNT}",
        container_name=STORAGE_CONTAINER,
        # 선택적: 특정 경로만 포함
        # path_prefix="docs/"
    )
)

print(f"📌 Blob Knowledge Source 정의:")
print(f"   이름: {BLOB_SOURCE_NAME}")
print(f"   Storage Account: {STORAGE_ACCOUNT}")
print(f"   Container: {STORAGE_CONTAINER}")
print(f"\n⚠️ 참고: 실제 생성은 유효한 Storage Account 정보가 필요합니다.")

### 4.3 두 번째 Search Index Knowledge Source

기존에 생성한 다른 인덱스를 추가 Knowledge Source로 연결합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchKnowledgeSourceData,
)

# 기존 인덱스 확인
print("📋 사용 가능한 인덱스 목록:")
print("=" * 50)

indexes = list(index_client.list_indexes())
for idx in indexes:
    has_semantic = idx.semantic_search is not None
    has_vector = any(f.type == "Collection(Edm.Single)" for f in idx.fields if hasattr(f, 'type'))
    
    print(f"\n📁 {idx.name}")
    print(f"   필드 수: {len(idx.fields)}")
    print(f"   시맨틱 검색: {'✅' if has_semantic else '❌'}")
    print(f"   벡터 검색: {'✅' if has_vector else '❌'}")

In [ ]:
# 추가 Search Index Knowledge Source (예: 다른 도메인의 인덱스)
# 이전 핸즈온에서 생성한 인덱스가 있다면 활용

ADDITIONAL_INDEX_NAME = "products-vector"  # 03-vector_search에서 생성한 인덱스
ADDITIONAL_SOURCE_NAME = "products-vector-source"

# 해당 인덱스가 존재하는지 확인
try:
    idx = index_client.get_index(ADDITIONAL_INDEX_NAME)
    
    # 시맨틱 설정 확인
    semantic_config = None
    if idx.semantic_search and idx.semantic_search.configurations:
        semantic_config = idx.semantic_search.configurations[0].name
    
    # 벡터 필드 확인
    vector_fields = [f.name for f in idx.fields if hasattr(f, 'vector_search_dimensions') and f.vector_search_dimensions]
    
    print(f"✅ 인덱스 '{ADDITIONAL_INDEX_NAME}' 발견")
    print(f"   시맨틱 설정: {semantic_config or '없음'}")
    print(f"   벡터 필드: {vector_fields or '없음'}")
    
    # Knowledge Source 생성 가능 여부
    if semantic_config:
        additional_source = KnowledgeSource(
            name=ADDITIONAL_SOURCE_NAME,
            description=f"{ADDITIONAL_INDEX_NAME} 인덱스 기반 Knowledge Source",
            data=SearchKnowledgeSourceData(
                index_name=ADDITIONAL_INDEX_NAME,
                semantic_configuration_name=semantic_config,
                vector_fields=vector_fields if vector_fields else None,
                content_fields=["content"],
                title_field="title"
            )
        )
        print(f"\n📌 추가 Knowledge Source 정의 완료")
    else:
        print("\n⚠️ 시맨틱 설정이 없어 Knowledge Source로 사용할 수 없습니다.")
        
except Exception as e:
    print(f"ℹ️ 인덱스 '{ADDITIONAL_INDEX_NAME}'가 없습니다: {e}")
    print("   03-vector_search 핸즈온을 먼저 완료하거나, 다른 인덱스를 사용하세요.")

## 5. Multi-Source Knowledge Base 생성

여러 Knowledge Source를 하나의 Knowledge Base로 통합합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeAgentAzureOpenAIModel,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMediumReasoningEffort,
)

MULTI_KB_NAME = "multi-source-kb"

# 사용 가능한 Knowledge Sources 수집
available_sources = []

# 기본 products-source (01에서 생성)
try:
    ks = index_client.get_knowledge_source("products-source")
    available_sources.append(KnowledgeSourceReference(
        name=ks.name,
        # 소스별 설정 커스터마이징
        include_references=True,
        include_reference_source_data=True
    ))
    print(f"✅ '{ks.name}' 추가됨")
except Exception as e:
    print(f"⚠️ products-source를 찾을 수 없습니다: {e}")

# 추가 소스가 있다면 추가
# available_sources.append(KnowledgeSourceReference(name=ADDITIONAL_SOURCE_NAME))

print(f"\n📊 총 {len(available_sources)}개의 Knowledge Source가 통합됩니다.")

In [ ]:
# LLM 모델 설정
agent_model = KnowledgeAgentAzureOpenAIModel(
    azure_open_ai_resource=aoai_endpoint.replace("https://", "").split(".")[0],
    azure_open_ai_deployment_id=chat_deployment,
    azure_open_ai_model_name=chat_deployment
)

# Multi-Source Knowledge Base 생성
if available_sources:
    multi_kb = KnowledgeBase(
        name=MULTI_KB_NAME,
        description="여러 데이터 소스를 통합한 Knowledge Base",
        knowledge_sources=available_sources,
        models=[agent_model],
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        retrieval_reasoning_effort=KnowledgeRetrievalMediumReasoningEffort(),
        # 고급 설정
        retrieval_instructions="사용자 질문에 가장 관련성 높은 정보를 우선적으로 검색하세요. 여러 소스의 정보를 종합하여 답변하세요."
    )
    
    # Knowledge Base 생성
    try:
        result = index_client.create_or_update_knowledge_base(multi_kb)
        print(f"✅ Multi-Source Knowledge Base '{result.name}' 생성 완료")
        print(f"   연결된 소스: {[s.name for s in result.knowledge_sources]}")
        print(f"   Output Mode: {result.output_mode}")
        print(f"   Reasoning Effort: Medium")
    except Exception as e:
        print(f"❌ 생성 실패: {e}")
else:
    print("⚠️ 사용 가능한 Knowledge Source가 없습니다.")

## 6. Multi-Source KB 테스트

In [ ]:
from azure.search.documents import KnowledgeBaseRetrievalClient
from azure.search.documents.models import KnowledgeBaseRetrievalRequest

# Multi-Source KB 클라이언트
try:
    multi_kb_client = KnowledgeBaseRetrievalClient(
        endpoint=search_endpoint,
        knowledge_base_name=MULTI_KB_NAME,
        credential=search_credential
    )
    
    print(f"✅ Multi-Source KB 클라이언트 생성 완료")
except Exception as e:
    print(f"⚠️ 클라이언트 생성 실패: {e}")
    multi_kb_client = None

In [ ]:
# 테스트 질의 실행
if multi_kb_client:
    test_query = "가격이 저렴하고 품질 좋은 아기 옷 추천해줘"
    
    print(f"📝 테스트 질의: {test_query}")
    print("=" * 60)
    
    request = KnowledgeBaseRetrievalRequest(
        query=test_query,
        top=5
    )
    
    try:
        response = multi_kb_client.retrieve(request)
        
        # 결과 출력
        if hasattr(response, 'data') and response.data:
            print(f"\n📄 반환된 문서 수: {len(response.data)}")
            for i, doc in enumerate(response.data, 1):
                print(f"\n[{i}] {doc.get('title', 'N/A')}")
                print(f"    소스: {doc.get('@search.source', 'N/A')}")
                print(f"    관련도: {doc.get('@search.score', 'N/A')}")
        
        # Activity 분석
        if hasattr(response, 'activity') and response.activity:
            print(f"\n🔍 Activity 수: {len(response.activity)}")
            for activity in response.activity:
                if hasattr(activity, 'knowledge_source'):
                    print(f"   - {activity.knowledge_source}에서 검색 수행")
                    
    except Exception as e:
        print(f"❌ 검색 실패: {e}")

## 7. Knowledge Source별 설정 커스터마이징

각 Knowledge Source에 대해 개별 설정을 적용할 수 있습니다.

In [ ]:
# Knowledge Source Reference의 커스터마이징 옵션
print("📋 Knowledge Source Reference 옵션:")
print("=" * 60)
print("""
KnowledgeSourceReference 파라미터:

┌─────────────────────────────────┬──────────────────────────────────────────┐
│ 파라미터                          │ 설명                                      │
├─────────────────────────────────┼──────────────────────────────────────────┤
│ name                            │ Knowledge Source 이름 (필수)              │
│ include_references              │ 참조 정보 포함 여부 (기본: True)           │
│ include_reference_source_data   │ 원본 데이터 포함 여부                      │
│ always_query_source             │ 항상 이 소스 쿼리 (필터링 방지)            │
│ max_sub_queries                 │ 최대 하위 쿼리 수                          │
│ reranker_threshold              │ Reranker 점수 임계값 (0.0-4.0)            │
└─────────────────────────────────┴──────────────────────────────────────────┘
""")

In [ ]:
# 소스별 커스터마이징 예시
customized_sources = [
    KnowledgeSourceReference(
        name="products-source",
        include_references=True,
        include_reference_source_data=True,
        always_query_source=True,  # 항상 쿼리
        max_sub_queries=5,
        reranker_threshold=0.5  # 낮은 점수 결과 필터링
    ),
    # 추가 소스 예시 (주석 처리)
    # KnowledgeSourceReference(
    #     name="secondary-source",
    #     include_references=True,
    #     reranker_threshold=0.3  # 더 관대한 필터링
    # )
]

print("📌 커스터마이징된 Knowledge Source 설정:")
for src in customized_sources:
    print(f"\n🔗 {src.name}")
    print(f"   always_query_source: {src.always_query_source}")
    print(f"   max_sub_queries: {src.max_sub_queries}")
    print(f"   reranker_threshold: {src.reranker_threshold}")

## 8. 생성된 리소스 요약

In [ ]:
print("\n" + "=" * 70)
print("📊 Multi-Source Knowledge Base 구성 요약")
print("=" * 70)

# Knowledge Sources 목록
print("\n🔗 등록된 Knowledge Sources:")
try:
    for ks in index_client.list_knowledge_sources():
        print(f"   - {ks.name}: {ks.description or 'No description'}")
except Exception as e:
    print(f"   조회 실패: {e}")

# Knowledge Bases 목록
print("\n🧠 등록된 Knowledge Bases:")
try:
    for kb in index_client.list_knowledge_bases():
        sources = [s.name for s in kb.knowledge_sources] if kb.knowledge_sources else []
        print(f"   - {kb.name}")
        print(f"     연결된 소스: {sources}")
        print(f"     Output Mode: {kb.output_mode}")
except Exception as e:
    print(f"   조회 실패: {e}")

## 9. 핵심 정리

### Multi-Source Knowledge Base 구성 시 고려사항

1. **소스 유형 선택**
   - `searchIndex`: 이미 인덱싱된 구조화된 데이터에 적합
   - `azureBlob`: 비정형 문서 (PDF, DOCX 등)에 적합
   - `web`: 온라인 문서, 최신 정보 필요 시 적합

2. **소스별 설정 최적화**
   - 중요 소스: `always_query_source=True`
   - 품질 우선: `reranker_threshold` 높게 설정
   - 범위 확대: `max_sub_queries` 높게 설정

3. **Output Mode 선택**
   - `extractive_data`: 원본 데이터 필요 시
   - `answer_synthesis`: LLM 생성 답변 필요 시 (Web 소스 필수)

### 다음 단계
- `05-runtime_parameters.ipynb`: 런타임 파라미터 튜닝
- `06-mcp_integration.ipynb`: MCP Tool 연동

In [ ]:
print("\n" + "=" * 70)
print("✅ Multi-Source Knowledge Base 구성 완료!")
print("=" * 70)
print("""
다음 노트북:
- 05-runtime_parameters.ipynb: Reranker Threshold, MaxSubQueries 튜닝
- 06-mcp_integration.ipynb: Foundry Agent Service + MCP 연동
""")